In [ ]:
import os

os.makedirs("docs", exist_ok=True)

docs = {
    "project_overview.md": """This project analyzes restaurant revenue trends using Google BigQuery and Looker Studio. The goal is to identify revenue drivers across day, time, and customer behavior.""",

    "insights.md": """Weekend dinner periods drive the majority of revenue.
Saturday dinner is the highest-performing segment.
Dinner generates roughly three times more revenue than lunch.
Friday is the weakest-performing day.
Average tip percentage is approximately 16.32%.""",

    "recommendations.md": """Increase staffing during weekend dinner periods.
Test promotions during weekday lunch.
Investigate low Friday performance.
Use upselling strategies during dinner peaks.""",

    "metric_definitions.md": """Total revenue = SUM(total_bill)
Tip percentage = tip / total_bill
Revenue by day shows performance across weekdays.
Revenue by time compares lunch vs dinner."""
}

for filename, content in docs.items():
    with open(f"docs/{filename}", "w") as f:
        f.write(content)

print("Docs created!")

Docs created!


In [ ]:
!pip install -q langchain langchain-openai chromadb tiktoken

In [ ]:
from langchain_core.documents import Document

print("LangChain import successful")

LangChain import successful


In [ ]:
import os
from langchain_core.documents import Document

raw_docs = []

for filename in os.listdir("docs"):
    if filename.endswith(".md"):
        with open(f"docs/{filename}", "r") as f:
            raw_docs.append(Document(page_content=f.read(), metadata={"source": filename}))

print(f"Loaded {len(raw_docs)} documents")

Loaded 4 documents


In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

split_docs = splitter.split_documents(raw_docs)

print(f"Created {len(split_docs)} chunks")

for d in split_docs[:2]:
    print(d.metadata, d.page_content[:150])
    print("----")

Created 4 chunks
{'source': 'project_overview.md'} This project analyzes restaurant revenue trends using Google BigQuery and Looker Studio. The goal is to identify revenue drivers across day, time, and
----
{'source': 'recommendations.md'} Increase staffing during weekend dinner periods.
Test promotions during weekday lunch.
Investigate low Friday performance.
Use upselling strategies du
----


In [ ]:
!pip install -q langchain-community chromadb

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
from langchain_community.vectorstores import Chroma

# Create vector DB
vectordb = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="chroma_db"
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("Vector DB ready 🚀")

Vector DB ready 🚀


In [ ]:
query = "What are the key insights from this project?"

results = retriever.invoke(query)

for doc in results:
    print(doc.metadata)
    print(doc.page_content)
    print("-----")

{'source': 'project_overview.md'}
This project analyzes restaurant revenue trends using Google BigQuery and Looker Studio. The goal is to identify revenue drivers across day, time, and customer behavior.
-----
{'source': 'insights.md'}
Weekend dinner periods drive the majority of revenue.
Saturday dinner is the highest-performing segment.
Dinner generates roughly three times more revenue than lunch.
Friday is the weakest-performing day.
Average tip percentage is approximately 16.32%.
-----
{'source': 'recommendations.md'}
Increase staffing during weekend dinner periods.
Test promotions during weekday lunch.
Investigate low Friday performance.
Use upselling strategies during dinner peaks.
-----


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [ ]:
def ask_question(query):
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    You are a data analyst assistant.

    Use the context below to answer the question.

    Context:
    {context}

    Question:
    {query}

    Answer clearly with insights and recommendations.
    """

    response = llm.invoke(prompt)
    return response.content

In [ ]:
ask_question("What should the restaurant do to increase revenue?")

'To increase revenue, the restaurant should focus on the following strategies, leveraging its strengths and addressing its weaknesses:\n\n**Key Insights:**\n\n*   **Dinner Dominance:** Dinner periods are significantly more profitable, generating roughly three times more revenue than lunch. Weekend dinners are the primary revenue driver, with Saturday dinner being the highest-performing segment.\n*   **Friday Weakness:** Friday is identified as the weakest-performing day.\n\n**Recommendations:**\n\n1.  **Maximize High-Performing Dinner Periods (especially Weekends):**\n    *   **Increase Staffing:** **Increase staffing during weekend dinner periods.** This ensures efficient service, reduces wait times, and improves customer experience during the busiest and most profitable times, preventing lost revenue due to operational bottlenecks.\n    *   **Implement Upselling:** **Use upselling strategies during dinner peaks.** Since dinner generates significantly more revenue, focusing upselling 

In [ ]:
ask_question("What are the key revenue drivers in this project?")

'Based on the provided context, the key revenue drivers for the restaurant are:\n\n**Key Revenue Drivers:**\n\n1.  **Dinner Periods:** Dinner service is a significant revenue driver, generating approximately three times more revenue than lunch.\n2.  **Weekend Periods:** Weekends, particularly dinner periods during the weekend, are crucial for revenue generation.\n3.  **Saturday Dinner:** This is identified as the single highest-performing segment, making it the most impactful revenue driver.\n\n**Insights and Recommendations:**\n\n*   **Focus on Dinner Optimization:** Given that dinner generates significantly more revenue than lunch, the restaurant should prioritize optimizing its dinner service. This could include menu enhancements, special dinner promotions, efficient staffing during dinner hours, and creating an appealing ambiance for evening diners.\n*   **Maximize Weekend Potential:** Weekends are prime revenue times. The restaurant should strategize to fully capitalize on this, e